# MMLAB PLOTS MULTIMODAL PLOT BONANZA
* Mutual information: Mutual information quantifies the amount of information obtained about one random variable through another random variable. It is a measure of dependency, linear or non-linear.

## The output


| Column | Description |
|--------|-------------|
| `filename` | Source file identifier |
| `condition_vision` | Vision/NoVision |
| `condition_movement` | Movement/NoMovement |
| `trial` | Trial identifier |
| `window_idx` | Window index (0, 1, 2, ...) |
| `window_start` | Window start time (s) |
| `window_end` | Window end time (s) |
| `var1` | First variable (P1) |
| `var2` | Second variable (P2) |
| `variable_pair_type` | Classification of pair |
| `max_mutual_info` | Peak MI across lags |
| `optimal_lag_mi` | Lag at peak MI (s) |
| `n_samples` | Samples in window |
| `sampling_rate` | Sampling frequency (Hz) |


In [5]:
# MMLAB MULTIMODAL Analysis - Mutual Information Only
"""
Multimodal coupling analysis using only Mutual Information.
"""

import pandas as pd
import numpy as np
import glob
import os
from scipy import stats
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import FancyArrowPatch
from scipy.signal import savgol_filter


def compute_mutual_information(x: np.ndarray, y: np.ndarray) -> float:
    """
    Compute mutual information between two time series using KDE.
    """
    # Clean data
    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]
    
    if len(x) < 2 or len(y) < 2:
        return 0.0
    
    # Standardize the data
    x = (x - np.mean(x)) / (np.std(x) + 1e-10)
    y = (y - np.mean(y)) / (np.std(y) + 1e-10)
    
    # Create KDE estimators
    kde_joint = stats.gaussian_kde(np.vstack([x, y]))
    kde_x = stats.gaussian_kde(x)
    kde_y = stats.gaussian_kde(y)
    
    # Sample points for numerical integration
    n_samples = 50
    x_range = np.linspace(min(x) - 1, max(x) + 1, n_samples)
    y_range = np.linspace(min(y) - 1, max(y) + 1, n_samples)
    X, Y = np.meshgrid(x_range, y_range)
    positions = np.vstack([X.ravel(), Y.ravel()])
    
    # Evaluate densities
    joint_density = kde_joint(positions).reshape(X.shape)
    x_density = kde_x(X[0, :])
    y_density = kde_y(Y[:, 0])
    X_density, Y_density = np.meshgrid(x_density, y_density)
    
    # Compute MI
    with np.errstate(divide='ignore', invalid='ignore'):
        mi_density = joint_density * np.log(joint_density / (X_density * Y_density + 1e-10))
    mi = np.nansum(mi_density) * (x_range[1] - x_range[0]) * (y_range[1] - y_range[0])
    
    return max(0, mi)  # Ensure non-negative MI


def compute_coupling_statistics(name, signal1, signal2, time):
    """
    Compute mutual information coupling statistics between two signals.
    
    Returns:
        dict with MI values at different lags
    """
    # Ensure inputs are numpy arrays
    signal1 = np.array(signal1)
    signal2 = np.array(signal2)
    time = np.array(time)

    # Check if inputs are scalar
    if signal1.ndim == 0 or signal2.ndim == 0 or time.ndim == 0:
        return {'name': name, 'max_mi': np.nan, 'optimal_lag': np.nan}

    # Check if inputs have the same length
    if len(signal1) != len(signal2) or len(signal1) != len(time):
        raise ValueError("signal1, signal2, and time must have the same length")

    # Normalize and center the data
    signal1 = (signal1 - np.min(signal1)) / (np.max(signal1) - np.min(signal1) + 1e-10)
    signal1 = signal1 - np.mean(signal1)
    signal2 = (signal2 - np.min(signal2)) / (np.max(signal2) - np.min(signal2) + 1e-10)
    signal2 = signal2 - np.mean(signal2)

    # Check if values are finite
    if not np.all(np.isfinite(signal1)) or not np.all(np.isfinite(signal2)):
        return {'name': name, 'max_mi': np.nan, 'optimal_lag': np.nan}

    # Compute sampling frequency
    fs = 1 / np.mean(np.diff(time))

    # Compute MI at different lags
    lags = [-0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3]
    lags_samples = [int(lag * fs) for lag in lags]
    mi_values = []
    
    for lag in lags_samples:
        if lag < 0:
            sig1_sub = signal1[:lag]
            sig2_sub = signal2[-lag:]
        elif lag > 0:
            sig1_sub = signal1[lag:]
            sig2_sub = signal2[:-lag]
        else:
            sig1_sub = signal1
            sig2_sub = signal2
            
        # Savitzky-Golay smoothing
        window = int(0.1 * fs)  # 100ms window
        if window < 5:
            window = 5  # Minimum for polyorder=2
        if window % 2 == 0:
            window += 1  # Must be odd
        
        if len(sig1_sub) > window and len(sig2_sub) > window:
            sig1_sub = savgol_filter(sig1_sub, window_length=window, polyorder=2)
            sig2_sub = savgol_filter(sig2_sub, window_length=window, polyorder=2)
        
        # Compute MI
        mi_value = compute_mutual_information(sig1_sub, sig2_sub)
        mi_values.append(mi_value)
    
    max_mi = np.max(mi_values)
    optimal_lag = lags[np.argmax(mi_values)]
    
    return {
        'name': name,
        'max_mi': max_mi,
        'optimal_lag': optimal_lag,
        'mi_values': mi_values,
        'lags': lags
    }


def classify_variable_pair(var1, var2):
    """
    Classify the type of variable pair for easier analysis.
    """
    modalities = []
    participants = []
    
    for var in [var1, var2]:
        if 'Amplitude_Envelope' in var:
            modalities.append('Audio')
        elif 'Filtered_ECG' in var or 'heart_rate' in var:
            modalities.append('ECG')
        elif 'Respiration' in var:
            modalities.append('Respiration')
        elif 'EMG' in var:
            modalities.append('EMG')
        elif 'right_index' in var:
            modalities.append('Motion')
            
        if '_P1' in var:
            participants.append('P1')
        elif '_P2' in var:
            participants.append('P2')
    
    # Classify pair type
    if participants[0] == participants[1]:
        pair_type = f'Within_{participants[0]}'
    else:
        pair_type = 'Between_Participants'
    
    if modalities[0] == modalities[1]:
        modality_type = f"Same_{modalities[0]}"
    else:
        modality_type = f"Cross_{modalities[0]}_{modalities[1]}"
    
    return f"{pair_type}_{modality_type}"


def get_modality_from_var(var_name):
    """Extract modality name from variable name."""
    if 'Amplitude_Envelope' in var_name:
        return 'Audio'
    elif 'Filtered_ECG' in var_name:
        return 'Heart Rate'
    elif 'Filtered_Respiration' in var_name:
        return 'Respiration'
    elif 'Filtered_EMG_Bicep' in var_name:
        return 'EMG Bicep'
    elif 'Filtered_EMG_Tricep' in var_name:
        return 'EMG Tricep'
    elif 'right_index_x' in var_name:
        return 'Motion X'
    elif 'right_index_y' in var_name:
        return 'Motion Y'
    elif 'right_index_z' in var_name:
        return 'Motion Z'
    else:
        return 'Unknown'


def calculate_p1_p2_coupling_stats(merged_folder='../merged_filteredtimeseries/', 
                                    output_file='p1_p2_coupling_statistics.csv'):
    """
    Calculate MI coupling statistics between P1 and P2 across all modalities.
    """
    
    # Find all merged CSV files
    csv_files = glob.glob(os.path.join(merged_folder, "*.csv"))
    print(f"Found {len(csv_files)} files to process")
    
    # Define P1-P2 variable pairs
    p1_modalities = [
        'Amplitude_Envelope_P1', 'Filtered_ECG_P1', 'Filtered_Respiration_P1', 
        'Filtered_EMG_Bicep_P1', 'Filtered_EMG_Tricep_P1', 
        'right_index_x_P1', 'right_index_y_P1', 'right_index_z_P1'
    ]
    p2_modalities = [
        'Amplitude_Envelope_P2', 'Filtered_ECG_P2', 'Filtered_Respiration_P2', 
        'Filtered_EMG_Bicep_P2', 'Filtered_EMG_Tricep_P2', 
        'right_index_x_P2', 'right_index_y_P2', 'right_index_z_P2'
    ]

    # Build all P1-P2 pairs
    variable_pairs = []
    for p1_var in p1_modalities:
        for p2_var in p2_modalities:
            variable_pairs.append((p1_var, p2_var))
    
    results = []
    
    for file_path in csv_files:
        print(f"\nProcessing: {os.path.basename(file_path)}")
        
        try:
            df = pd.read_csv(file_path)
            filename = os.path.basename(file_path)
            
            # Extract conditions
            condition_vision = 'Vision' if 'NoVision' not in filename else 'NoVision'
            condition_movement = 'Movement' if 'NoMovement' not in filename else 'NoMovement'
            trial = df['Trial'].iloc[0] if 'Trial' in df.columns else 'Unknown'
            
            time = df['Time'].values
            
            # Check minimum duration
            if time.max() - time.min() < 5.0:
                print(f"  Skipping: insufficient duration ({time.max() - time.min():.2f}s)")
                continue
            
            # Sliding window parameters
            window_duration = 5.0
            step_size = 1.0
            n_windows = int((time.max() - time.min() - window_duration) / step_size) + 1
            
            print(f"  Analyzing {n_windows} windows")
            
            for var1, var2 in variable_pairs:
                if var1 not in df.columns or var2 not in df.columns:
                    continue
                
                for window_idx in range(n_windows):
                    window_start = time.min() + window_idx * step_size
                    window_end = window_start + window_duration
                    
                    window_mask = (time >= window_start) & (time <= window_end)
                    time_window = time[window_mask]
                    var1_window = df[var1].values[window_mask]
                    var2_window = df[var2].values[window_mask]
                    
                    # Skip if insufficient data
                    if (len(time_window) < 100 or 
                        np.sum(~np.isnan(var1_window)) < 50 or 
                        np.sum(~np.isnan(var2_window)) < 50):
                        continue
                    
                    try:
                        stats_result = compute_coupling_statistics(
                            filename, var1_window, var2_window, time_window
                        )
                        
                        result = {
                            'filename': filename,
                            'condition_vision': condition_vision,
                            'condition_movement': condition_movement,
                            'trial': trial,
                            'window_idx': window_idx,
                            'window_start': window_start,
                            'window_end': window_end,
                            'var1': var1,
                            'var2': var2,
                            'variable_pair_type': classify_variable_pair(var1, var2),
                            'max_mutual_info': stats_result['max_mi'],
                            'optimal_lag_mi': stats_result['optimal_lag'],
                            'n_samples': len(time_window),
                            'sampling_rate': 1 / np.mean(np.diff(time_window))
                        }
                        
                        results.append(result)
                        
                    except Exception as e:
                        print(f"      Error in window {window_idx}: {e}")
                        continue
                        
        except Exception as e:
            print(f"  Error processing {filename}: {e}")
            continue
    
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv(output_file, index=False)
        print(f"\n✓ Saved {len(results_df)} coupling statistics to {output_file}")
        
        print(f"\nSummary:")
        print(f"  Files processed: {results_df['filename'].nunique()}")
        print(f"  Variable pairs: {results_df['variable_pair_type'].nunique()}")
        print(f"  Total windows: {len(results_df)}")
        
        return results_df
    else:
        print("No results generated")
        return pd.DataFrame()

In [ ]:
# Parralelized

In [1]:
import pandas as pd
import numpy as np
import glob
import os
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
from tqdm import tqdm
import multiprocessing


def compute_mutual_information(x: np.ndarray, y: np.ndarray) -> float:
    """
    Compute mutual information between two time series using KDE.
    """
    # Clean data
    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]
    
    if len(x) < 2 or len(y) < 2:
        return 0.0
    
    # Standardize the data
    x = (x - np.mean(x)) / (np.std(x) + 1e-10)
    y = (y - np.mean(y)) / (np.std(y) + 1e-10)
    
    # Create KDE estimators
    try:
        kde_joint = stats.gaussian_kde(np.vstack([x, y]))
        kde_x = stats.gaussian_kde(x)
        kde_y = stats.gaussian_kde(y)
    except np.linalg.LinAlgError:
        return 0.0  # Singular matrix, return 0
    
    # Sample points for numerical integration
    n_samples = 50
    x_range = np.linspace(min(x) - 1, max(x) + 1, n_samples)
    y_range = np.linspace(min(y) - 1, max(y) + 1, n_samples)
    X, Y = np.meshgrid(x_range, y_range)
    positions = np.vstack([X.ravel(), Y.ravel()])
    
    # Evaluate densities
    joint_density = kde_joint(positions).reshape(X.shape)
    x_density = kde_x(X[0, :])
    y_density = kde_y(Y[:, 0])
    X_density, Y_density = np.meshgrid(x_density, y_density)
    
    # Compute MI
    with np.errstate(divide='ignore', invalid='ignore'):
        mi_density = joint_density * np.log(joint_density / (X_density * Y_density + 1e-10))
    mi = np.nansum(mi_density) * (x_range[1] - x_range[0]) * (y_range[1] - y_range[0])
    
    return max(0, mi)


def compute_coupling_statistics(signal1, signal2, fs):
    """
    Compute mutual information coupling statistics between two signals.
    Simplified version for parallel processing.
    """
    # Normalize and center
    signal1 = (signal1 - np.min(signal1)) / (np.max(signal1) - np.min(signal1) + 1e-10)
    signal1 = signal1 - np.mean(signal1)
    signal2 = (signal2 - np.min(signal2)) / (np.max(signal2) - np.min(signal2) + 1e-10)
    signal2 = signal2 - np.mean(signal2)

    if not np.all(np.isfinite(signal1)) or not np.all(np.isfinite(signal2)):
        return np.nan, np.nan

    # Compute MI at different lags
    lags = [-0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3]
    lags_samples = [int(lag * fs) for lag in lags]
    mi_values = []
    
    # Savgol window setup
    window = int(0.1 * fs)
    if window < 5:
        window = 5
    if window % 2 == 0:
        window += 1
    
    for lag in lags_samples:
        if lag < 0:
            sig1_sub = signal1[:lag]
            sig2_sub = signal2[-lag:]
        elif lag > 0:
            sig1_sub = signal1[lag:]
            sig2_sub = signal2[:-lag]
        else:
            sig1_sub = signal1.copy()
            sig2_sub = signal2.copy()
        
        if len(sig1_sub) > window and len(sig2_sub) > window:
            sig1_sub = savgol_filter(sig1_sub, window_length=window, polyorder=2)
            sig2_sub = savgol_filter(sig2_sub, window_length=window, polyorder=2)
        
        mi_value = compute_mutual_information(sig1_sub, sig2_sub)
        mi_values.append(mi_value)
    
    max_mi = np.max(mi_values)
    optimal_lag = lags[np.argmax(mi_values)]
    
    return max_mi, optimal_lag


def process_single_window(args):
    """
    Process a single window for a single variable pair.
    Designed for parallel execution.
    """
    (var1_window, var2_window, time_window, var1, var2, 
     filename, condition_vision, condition_movement, trial, window_idx,
     window_start, window_end) = args
    
    # Skip if insufficient data
    if (len(time_window) < 100 or 
        np.sum(~np.isnan(var1_window)) < 50 or 
        np.sum(~np.isnan(var2_window)) < 50):
        return None
    
    try:
        fs = 1 / np.mean(np.diff(time_window))
        max_mi, optimal_lag = compute_coupling_statistics(var1_window, var2_window, fs)
        
        if np.isnan(max_mi):
            return None
            
        return {
            'filename': filename,
            'condition_vision': condition_vision,
            'condition_movement': condition_movement,
            'trial': trial,
            'window_idx': window_idx,
            'window_start': window_start,
            'window_end': window_end,
            'var1': var1,
            'var2': var2,
            'variable_pair_type': classify_variable_pair(var1, var2),
            'max_mutual_info': max_mi,
            'optimal_lag_mi': optimal_lag,
            'n_samples': len(time_window),
            'sampling_rate': fs
        }
    except Exception:
        return None


def classify_variable_pair(var1, var2):
    """Classify the type of variable pair."""
    modalities = []
    participants = []
    
    for var in [var1, var2]:
        if 'Amplitude_Envelope' in var:
            modalities.append('Audio')
        elif 'Filtered_ECG' in var or 'heart_rate' in var:
            modalities.append('ECG')
        elif 'Respiration' in var:
            modalities.append('Respiration')
        elif 'EMG' in var:
            modalities.append('EMG')
        elif 'right_index' in var:
            modalities.append('Motion')
        else:
            modalities.append('Other')
            
        if '_P1' in var:
            participants.append('P1')
        elif '_P2' in var:
            participants.append('P2')
        else:
            participants.append('Unknown')
    
    if len(participants) == 2 and participants[0] == participants[1]:
        pair_type = f'Within_{participants[0]}'
    else:
        pair_type = 'Between_Participants'
    
    if len(modalities) == 2 and modalities[0] == modalities[1]:
        modality_type = f"Same_{modalities[0]}"
    else:
        modality_type = f"Cross_{modalities[0]}_{modalities[1]}"
    
    return f"{pair_type}_{modality_type}"


def get_modality_from_var(var_name):
    """Extract modality name from variable name."""
    if 'Amplitude_Envelope' in var_name:
        return 'Audio'
    elif 'Filtered_ECG' in var_name:
        return 'Heart Rate'
    elif 'Filtered_Respiration' in var_name:
        return 'Respiration'
    elif 'Filtered_EMG_Bicep' in var_name:
        return 'EMG Bicep'
    elif 'Filtered_EMG_Tricep' in var_name:
        return 'EMG Tricep'
    elif 'right_index_x' in var_name:
        return 'Motion X'
    elif 'right_index_y' in var_name:
        return 'Motion Y'
    elif 'right_index_z' in var_name:
        return 'Motion Z'
    else:
        return 'Unknown'


def calculate_p1_p2_coupling_stats_parallel(
    merged_folder='../merged_filteredtimeseries/', 
    output_file='p1_p2_coupling_statistics.csv',
    n_jobs=-1,  # -1 uses all available cores
    window_duration=5.0,
    step_size=1.0
):
    """
    Calculate MI coupling statistics between P1 and P2 across all modalities.
    Parallelized version.
    
    Parameters:
    -----------
    merged_folder : str
        Path to folder containing merged CSV files
    output_file : str
        Output CSV filename
    n_jobs : int
        Number of parallel jobs (-1 = all cores, -2 = all but one)
    window_duration : float
        Sliding window duration in seconds
    step_size : float
        Sliding window step size in seconds
    """
    
    # Determine number of cores
    if n_jobs == -1:
        n_cores = multiprocessing.cpu_count()
    elif n_jobs == -2:
        n_cores = max(1, multiprocessing.cpu_count() - 1)
    else:
        n_cores = n_jobs
    
    print(f"Using {n_cores} CPU cores for parallel processing")
    
    # Find all merged CSV files
    csv_files = glob.glob(os.path.join(merged_folder, "*.csv"))
    print(f"Found {len(csv_files)} files to process")
    
    if len(csv_files) == 0:
        print("No CSV files found!")
        return pd.DataFrame()
    
    # Define P1-P2 variable pairs
    p1_modalities = [
        'Amplitude_Envelope_P1', 'Filtered_ECG_P1', 'Filtered_Respiration_P1', 
        'Filtered_EMG_Bicep_P1', 'Filtered_EMG_Tricep_P1', 
        'right_index_x_P1', 'right_index_y_P1', 'right_index_z_P1'
    ]
    p2_modalities = [
        'Amplitude_Envelope_P2', 'Filtered_ECG_P2', 'Filtered_Respiration_P2', 
        'Filtered_EMG_Bicep_P2', 'Filtered_EMG_Tricep_P2', 
        'right_index_x_P2', 'right_index_y_P2', 'right_index_z_P2'
    ]

    variable_pairs = [(p1, p2) for p1 in p1_modalities for p2 in p2_modalities]
    
    # Collect all tasks
    print("Preparing tasks...")
    all_tasks = []
    
    for file_path in tqdm(csv_files, desc="Loading files"):
        try:
            df = pd.read_csv(file_path)
            filename = os.path.basename(file_path)
            
            condition_vision = 'Vision' if 'NoVision' not in filename else 'NoVision'
            condition_movement = 'Movement' if 'NoMovement' not in filename else 'NoMovement'
            trial = df['Trial'].iloc[0] if 'Trial' in df.columns else 'Unknown'
            
            time = df['Time'].values
            
            if time.max() - time.min() < window_duration:
                continue
            
            n_windows = int((time.max() - time.min() - window_duration) / step_size) + 1
            
            for var1, var2 in variable_pairs:
                if var1 not in df.columns or var2 not in df.columns:
                    continue
                
                # Pre-extract all data for this variable pair
                var1_data = df[var1].values
                var2_data = df[var2].values
                
                for window_idx in range(n_windows):
                    window_start = time.min() + window_idx * step_size
                    window_end = window_start + window_duration
                    
                    window_mask = (time >= window_start) & (time <= window_end)
                    
                    task = (
                        var1_data[window_mask].copy(),
                        var2_data[window_mask].copy(),
                        time[window_mask].copy(),
                        var1, var2,
                        filename, condition_vision, condition_movement, trial,
                        window_idx, window_start, window_end
                    )
                    all_tasks.append(task)
                    
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            continue
    
    print(f"\nTotal tasks to process: {len(all_tasks)}")
    
    if len(all_tasks) == 0:
        print("No tasks to process!")
        return pd.DataFrame()
    
    # Process in parallel with progress bar
    print(f"Processing {len(all_tasks)} windows across {n_cores} cores...")
    
    results = Parallel(n_jobs=n_cores, verbose=0)(
        delayed(process_single_window)(task) 
        for task in tqdm(all_tasks, desc="Computing MI")
    )
    
    # Filter out None results
    results = [r for r in results if r is not None]
    
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv(output_file, index=False)
        print(f"\n✓ Saved {len(results_df)} coupling statistics to {output_file}")
        
        print(f"\nSummary:")
        print(f"  Files processed: {results_df['filename'].nunique()}")
        print(f"  Variable pairs: {results_df['variable_pair_type'].nunique()}")
        print(f"  Total windows: {len(results_df)}")
        print(f"  Success rate: {len(results_df)}/{len(all_tasks)} ({100*len(results_df)/len(all_tasks):.1f}%)")
        
        return results_df
    else:
        print("No results generated")
        return pd.DataFrame()


# =============================================================================
# ALTERNATIVE: Batch processing by file (better for memory)
# =============================================================================

def process_single_file(file_path, variable_pairs, window_duration=5.0, step_size=1.0):
    """
    Process a single file - all variable pairs and windows.
    Better for memory management with large files.
    """
    results = []
    
    try:
        df = pd.read_csv(file_path)
        filename = os.path.basename(file_path)
        
        condition_vision = 'Vision' if 'NoVision' not in filename else 'NoVision'
        condition_movement = 'Movement' if 'NoMovement' not in filename else 'NoMovement'
        trial = df['Trial'].iloc[0] if 'Trial' in df.columns else 'Unknown'
        
        time = df['Time'].values
        
        if time.max() - time.min() < window_duration:
            return results
        
        n_windows = int((time.max() - time.min() - window_duration) / step_size) + 1
        fs = 1 / np.mean(np.diff(time))
        
        for var1, var2 in variable_pairs:
            if var1 not in df.columns or var2 not in df.columns:
                continue
            
            var1_data = df[var1].values
            var2_data = df[var2].values
            
            for window_idx in range(n_windows):
                window_start = time.min() + window_idx * step_size
                window_end = window_start + window_duration
                
                window_mask = (time >= window_start) & (time <= window_end)
                time_window = time[window_mask]
                var1_window = var1_data[window_mask]
                var2_window = var2_data[window_mask]
                
                if (len(time_window) < 100 or 
                    np.sum(~np.isnan(var1_window)) < 50 or 
                    np.sum(~np.isnan(var2_window)) < 50):
                    continue
                
                try:
                    max_mi, optimal_lag = compute_coupling_statistics(
                        var1_window, var2_window, fs
                    )
                    
                    if not np.isnan(max_mi):
                        results.append({
                            'filename': filename,
                            'condition_vision': condition_vision,
                            'condition_movement': condition_movement,
                            'trial': trial,
                            'window_idx': window_idx,
                            'window_start': window_start,
                            'window_end': window_end,
                            'var1': var1,
                            'var2': var2,
                            'variable_pair_type': classify_variable_pair(var1, var2),
                            'max_mutual_info': max_mi,
                            'optimal_lag_mi': optimal_lag,
                            'n_samples': len(time_window),
                            'sampling_rate': fs
                        })
                except Exception:
                    continue
                    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
    
    return results


def calculate_p1_p2_coupling_stats_parallel_byfile(
    merged_folder='../merged_filteredtimeseries/', 
    output_file='p1_p2_coupling_statistics.csv',
    n_jobs=-1,
    window_duration=5.0,
    step_size=1.0
):
    """
    Parallelized by file (more memory efficient for large datasets).
    """
    if n_jobs == -1:
        n_cores = multiprocessing.cpu_count()
    elif n_jobs == -2:
        n_cores = max(1, multiprocessing.cpu_count() - 1)
    else:
        n_cores = n_jobs
    
    print(f"Using {n_cores} CPU cores (file-level parallelization)")
    
    csv_files = glob.glob(os.path.join(merged_folder, "*.csv"))
    print(f"Found {len(csv_files)} files to process")
    
    if len(csv_files) == 0:
        return pd.DataFrame()
    
    # Define variable pairs
    p1_modalities = [
        'Amplitude_Envelope_P1', 'Filtered_ECG_P1', 'Filtered_Respiration_P1', 
        'Filtered_EMG_Bicep_P1', 'Filtered_EMG_Tricep_P1', 
        'right_index_x_P1', 'right_index_y_P1', 'right_index_z_P1'
    ]
    p2_modalities = [
        'Amplitude_Envelope_P2', 'Filtered_ECG_P2', 'Filtered_Respiration_P2', 
        'Filtered_EMG_Bicep_P2', 'Filtered_EMG_Tricep_P2', 
        'right_index_x_P2', 'right_index_y_P2', 'right_index_z_P2'
    ]
    variable_pairs = [(p1, p2) for p1 in p1_modalities for p2 in p2_modalities]
    
    # Process files in parallel
    print(f"Processing {len(csv_files)} files...")
    
    all_results = Parallel(n_jobs=n_cores, verbose=10)(
        delayed(process_single_file)(f, variable_pairs, window_duration, step_size)
        for f in csv_files
    )
    
    # Flatten results
    results = [r for file_results in all_results for r in file_results]
    
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv(output_file, index=False)
        print(f"\n✓ Saved {len(results_df)} results to {output_file}")
        return results_df
    else:
        print("No results generated")
        return pd.DataFrame()

# Plotting
Three types of plots are generated.

* Overall coupling plots

* Cross-modality coupling per condition



In [2]:
# Consistent color scheme for conditions
CONDITION_COLORS = {
    'NoVision x Movement': '#e41a1c',
    'NoVision x NoMovement': '#377eb8',
    'Vision x Movement': '#4daf4a',
    'Vision x NoMovement': '#984ea3'
}

def create_modality_overview_plots(results_df, output_folder='coupling_plots/'):
    """
    Create MI overview plots for each modality's P1-P2 coupling.
    """
    os.makedirs(output_folder, exist_ok=True)
    
    # Get P1-P2 same-modality pairs
    p1_p2_data = results_df[
        (results_df['var1'].str.contains('_P1')) &
        (results_df['var2'].str.contains('_P2'))
    ].copy()
    
    p1_p2_data['modality1'] = p1_p2_data['var1'].apply(get_modality_from_var)
    p1_p2_data['modality2'] = p1_p2_data['var2'].apply(get_modality_from_var)
    
    same_modality = p1_p2_data[p1_p2_data['modality1'] == p1_p2_data['modality2']]
    modalities = same_modality['modality1'].unique()
    
    print(f"\nCreating overview plots for {len(modalities)} modalities")
    
    for modality in modalities:
        mod_data = same_modality[same_modality['modality1'] == modality].copy()
        
        if len(mod_data) == 0:
            continue
        
        mod_data['condition'] = (mod_data['condition_vision'] + ' x ' + 
                                  mod_data['condition_movement'])
        conditions = sorted(mod_data['condition'].unique())
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Panel 1: MI Distribution by Condition (Violin + Scatter)
        ax1 = axes[0]
        data_by_cond = [mod_data[mod_data['condition'] == c]['max_mutual_info'].dropna().values 
                        for c in conditions]
        
        if all(len(d) > 0 for d in data_by_cond):
            parts = ax1.violinplot(data_by_cond, positions=range(len(conditions)),
                                   showmeans=False, showmedians=True, showextrema=False)
            
            for pc in parts['bodies']:
                pc.set_facecolor('lightcoral')
                pc.set_alpha(0.7)
            
            for i, cond in enumerate(conditions):
                x = np.random.normal(i, 0.04, size=len(data_by_cond[i]))
                color = CONDITION_COLORS.get(cond, 'gray')
                ax1.scatter(x, data_by_cond[i], alpha=0.7, s=40, color=color)
        
        ax1.set_xticks(range(len(conditions)))
        ax1.set_xticklabels(conditions, rotation=45, ha='right', fontsize=12, fontweight='bold')
        ax1.set_ylabel('Mutual Information', fontsize=14, fontweight='bold')
        ax1.set_title(f'MI Distribution: {modality} P1-P2', fontsize=16, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        
        # Panel 2: Optimal Lag Distribution
        ax2 = axes[1]
        lag_data = [mod_data[mod_data['condition'] == c]['optimal_lag_mi'].dropna().values 
                    for c in conditions]
        
        if all(len(d) > 0 for d in lag_data):
            parts = ax2.violinplot(lag_data, positions=range(len(conditions)),
                                   showmeans=False, showmedians=True, showextrema=False)
            
            for pc in parts['bodies']:
                pc.set_facecolor('lightblue')
                pc.set_alpha(0.7)
            
            for i, cond in enumerate(conditions):
                x = np.random.normal(i, 0.04, size=len(lag_data[i]))
                color = CONDITION_COLORS.get(cond, 'gray')
                ax2.scatter(x, lag_data[i], alpha=0.7, s=40, color=color)
        
        ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax2.set_xticks(range(len(conditions)))
        ax2.set_xticklabels(conditions, rotation=45, ha='right', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Optimal Lag (s)', fontsize=14, fontweight='bold')
        ax2.set_title(f'Optimal Lag: {modality} P1-P2', fontsize=16, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        safe_modality = modality.replace(' ', '_').replace('/', '_')
        plt.savefig(os.path.join(output_folder, f'{safe_modality}_MI_overview.png'), 
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.savefig(os.path.join(output_folder, f'{safe_modality}_MI_overview.svg'), 
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        
        print(f"  ✓ {modality}")


def create_cross_modality_matrices(results_df, output_folder='coupling_plots/'):
    """
    Create cross-modality MI matrices for each condition.
    """
    os.makedirs(output_folder, exist_ok=True)
    
    p1_p2_data = results_df[
        (results_df['var1'].str.contains('_P1')) &
        (results_df['var2'].str.contains('_P2'))
    ].copy()
    
    if len(p1_p2_data) == 0:
        print("No P1-P2 data found")
        return
    
    p1_p2_data['modality1'] = p1_p2_data['var1'].apply(get_modality_from_var)
    p1_p2_data['modality2'] = p1_p2_data['var2'].apply(get_modality_from_var)
    
    modalities = sorted(list(set(p1_p2_data['modality1'].unique()) | 
                              set(p1_p2_data['modality2'].unique())))
    modalities = [m for m in modalities if m != 'Unknown']
    
    conditions = p1_p2_data.groupby(['condition_vision', 'condition_movement']).size().reset_index()
    
    # Global scaling
    all_values = p1_p2_data['max_mutual_info'].dropna().values
    global_vmin = 0
    global_vmax = np.percentile(all_values, 95) if len(all_values) > 0 else 2.0
    
    print(f"\nCreating cross-modality matrices (scale: {global_vmin:.3f} - {global_vmax:.3f})")
    
    for _, cond_row in conditions.iterrows():
        vision = cond_row['condition_vision']
        movement = cond_row['condition_movement']
        cond_name = f"{vision} x {movement}"
        
        cond_data = p1_p2_data[
            (p1_p2_data['condition_vision'] == vision) & 
            (p1_p2_data['condition_movement'] == movement)
        ]
        
        # Build matrix
        matrix = np.full((len(modalities), len(modalities)), np.nan)
        
        for i, p2_mod in enumerate(modalities):
            for j, p1_mod in enumerate(modalities):
                pair_data = cond_data[
                    (cond_data['modality1'] == p1_mod) & 
                    (cond_data['modality2'] == p2_mod)
                ]
                if len(pair_data) > 0:
                    matrix[i, j] = pair_data['max_mutual_info'].mean()
        
        # Plot
        fig, ax = plt.subplots(figsize=(16, 14))
        
        mask = np.isnan(matrix)
        n_mod = len(modalities)
        annot_size = 24 if n_mod <= 5 else (20 if n_mod <= 7 else 16)
        
        sns.heatmap(matrix, annot=True, fmt='.3f', cmap='viridis',
                    mask=mask, cbar_kws={'label': 'Mutual Information', 'shrink': 0.6},
                    xticklabels=modalities, yticklabels=modalities,
                    ax=ax, square=True, vmin=global_vmin, vmax=global_vmax,
                    annot_kws={'fontsize': annot_size, 'fontweight': 'bold'},
                    linewidths=2.0, linecolor='white')
        
        ax.set_xlabel('P1 Modality', fontsize=24, fontweight='bold', labelpad=20)
        ax.set_ylabel('P2 Modality', fontsize=24, fontweight='bold', labelpad=20)
        ax.set_title(f'P1-P2 Cross-Modality MI - {cond_name}', fontsize=28, fontweight='bold', pad=40)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=18, fontweight='bold')
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=18, fontweight='bold')
        
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=16)
        cbar.set_label('Mutual Information', fontsize=20, fontweight='bold')
        
        plt.tight_layout()
        
        safe_cond = cond_name.replace(' ', '_').replace('x', 'vs')
        plt.savefig(os.path.join(output_folder, f'cross_modality_MI_matrix_{safe_cond}.png'),
                    dpi=300, facecolor='white')
        plt.savefig(os.path.join(output_folder, f'cross_modality_MI_matrix_{safe_cond}.svg'),
                    dpi=300, facecolor='white')
        plt.close()
        
        print(f"  ✓ {cond_name}")


def create_audio_vs_modality_scatters(results_df, output_folder='coupling_plots/'):
    """
    Create scatter plots of Audio P1-P2 MI vs each other modality's P1-P2 MI.
    """
    os.makedirs(output_folder, exist_ok=True)
    
    # Get envelope coupling
    envelope_data = results_df[
        (results_df['var1'] == 'Amplitude_Envelope_P1') & 
        (results_df['var2'] == 'Amplitude_Envelope_P2')
    ].copy()
    
    if len(envelope_data) == 0:
        print("No envelope data found")
        return
    
    envelope_avg = envelope_data.groupby(
        ['trial', 'condition_vision', 'condition_movement', 'window_idx']
    )['max_mutual_info'].mean().reset_index()
    
    # Find other P1-P2 same-modality pairs
    p1_vars = [v for v in results_df['var1'].unique() if '_P1' in v and 'Amplitude_Envelope' not in v]
    
    conditions = envelope_avg.groupby(['condition_vision', 'condition_movement']).size().reset_index()
    
    print(f"\nCreating Audio vs Modality scatter plots")
    
    for p1_var in p1_vars:
        p1_base = p1_var.replace('_P1', '')
        p2_var = p1_base + '_P2'
        modality = get_modality_from_var(p1_var)
        
        mod_data = results_df[
            (results_df['var1'] == p1_var) & 
            (results_df['var2'] == p2_var)
        ]
        
        if len(mod_data) == 0:
            continue
        
        for _, cond_row in conditions.iterrows():
            vision = cond_row['condition_vision']
            movement = cond_row['condition_movement']
            cond_name = f"{vision} x {movement}"
            
            fig, ax = plt.subplots(figsize=(8, 8))
            
            mod_cond = mod_data[
                (mod_data['condition_vision'] == vision) &
                (mod_data['condition_movement'] == movement)
            ]
            
            if len(mod_cond) == 0:
                ax.text(0.5, 0.5, f'No data for {cond_name}', 
                        ha='center', va='center', transform=ax.transAxes, fontsize=14)
            else:
                mod_avg = mod_cond.groupby(['trial', 'window_idx'])['max_mutual_info'].mean().reset_index()
                
                env_cond = envelope_avg[
                    (envelope_avg['condition_vision'] == vision) &
                    (envelope_avg['condition_movement'] == movement)
                ]
                
                merged = pd.merge(env_cond, mod_avg, on=['trial', 'window_idx'], suffixes=('_env', '_mod'))
                
                if len(merged) > 2:
                    x = merged['max_mutual_info_env'].values
                    y = merged['max_mutual_info_mod'].values
                    
                    mask = ~(np.isnan(x) | np.isnan(y))
                    x, y = x[mask], y[mask]
                    
                    if len(x) > 2:
                        color = CONDITION_COLORS.get(cond_name, 'gray')
                        ax.scatter(x, y, alpha=0.7, s=100, color=color, 
                                   edgecolors='black', linewidth=0.8)
                        
                        # Regression line
                        slope, intercept, r, p, _ = stats.linregress(x, y)
                        line_x = np.array([x.min(), x.max()])
                        ax.plot(line_x, slope * line_x + intercept, 'k--', linewidth=3, alpha=0.8)
                        
                        ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.3f}\nn = {len(x)}',
                                transform=ax.transAxes, verticalalignment='top',
                                fontsize=12, fontweight='bold',
                                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
            
            ax.set_xlabel('MI (Audio P1-P2)', fontsize=16, fontweight='bold')
            ax.set_ylabel(f'MI ({modality} P1-P2)', fontsize=16, fontweight='bold')
            ax.set_title(f'{cond_name}', fontsize=18, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.tick_params(axis='both', labelsize=12)
            
            plt.tight_layout()
            
            safe_mod = modality.replace(' ', '_').replace('/', '_')
            safe_cond = cond_name.replace(' ', '_').replace('x', 'vs')
            plt.savefig(os.path.join(output_folder, f'audio_vs_{safe_mod}_{safe_cond}_MI.png'),
                        dpi=300, bbox_inches='tight', facecolor='white')
            plt.savefig(os.path.join(output_folder, f'audio_vs_{safe_mod}_{safe_cond}_MI.svg'),
                        dpi=300, bbox_inches='tight', facecolor='white')
            plt.close()
        
        print(f"  ✓ {modality}")


def create_network_plots(results_df, output_folder='coupling_plots/'):
    """
    Create network visualizations of P1-P2 modality coupling with jitter plots.
    """
    os.makedirs(output_folder, exist_ok=True)
    
    p1_p2_data = results_df[
        (results_df['var1'].str.contains('_P1')) &
        (results_df['var2'].str.contains('_P2'))
    ].copy()
    
    if len(p1_p2_data) == 0:
        print("No P1-P2 data found")
        return
    
    p1_p2_data['modality1'] = p1_p2_data['var1'].apply(get_modality_from_var)
    p1_p2_data['modality2'] = p1_p2_data['var2'].apply(get_modality_from_var)
    
    conditions = p1_p2_data.groupby(['condition_vision', 'condition_movement']).size().reset_index()
    
    # Global MI range
    all_mi = p1_p2_data['max_mutual_info'].dropna().values
    global_mi_min = 0
    global_mi_max = np.percentile(all_mi, 95) if len(all_mi) > 0 else 2.0
    
    print(f"\nCreating network plots (MI range: {global_mi_min:.3f} - {global_mi_max:.3f})")
    
    for _, cond_row in conditions.iterrows():
        vision = cond_row['condition_vision']
        movement = cond_row['condition_movement']
        cond_name = f"{vision} x {movement}"
        
        cond_data = p1_p2_data[
            (p1_p2_data['condition_vision'] == vision) & 
            (p1_p2_data['condition_movement'] == movement)
        ]
        
        if len(cond_data) == 0:
            continue
        
        fig, (ax_net, ax_jitter) = plt.subplots(1, 2, figsize=(24, 12))
        
        # === Network Plot ===
        G = nx.DiGraph()
        
        modalities = sorted(list(set(cond_data['modality1'].unique()) | 
                                  set(cond_data['modality2'].unique())))
        modalities = [m for m in modalities if m != 'Unknown']
        
        # Add nodes
        for mod in modalities:
            G.add_node(f"{mod}_P1", participant='P1', modality=mod)
            G.add_node(f"{mod}_P2", participant='P2', modality=mod)
        
        # Add edges
        for _, row in cond_data.iterrows():
            p1_mod, p2_mod = row['modality1'], row['modality2']
            mi_val = row['max_mutual_info']
            
            if not np.isnan(mi_val) and p1_mod != 'Unknown' and p2_mod != 'Unknown':
                G.add_edge(f"{p1_mod}_P1", f"{p2_mod}_P2", weight=mi_val)
        
        # Layout
        pos = {}
        y_positions = np.linspace(1, 0, len(modalities))
        for i, mod in enumerate(modalities):
            pos[f"{mod}_P1"] = (0, y_positions[i])
            pos[f"{mod}_P2"] = (1, y_positions[i])
        
        # Draw nodes
        node_colors = ['lightblue' if '_P1' in n else 'lightcoral' for n in G.nodes()]
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=3000,
                               ax=ax_net, alpha=0.8, edgecolors='black', linewidths=2)
        
        # Draw edges with MI-based styling
        edges = list(G.edges())
        if edges:
            weights = [G[u][v]['weight'] for u, v in edges]
            
            for i, (u, v) in enumerate(edges):
                w = weights[i]
                norm_w = (w - global_mi_min) / (global_mi_max - global_mi_min + 1e-10)
                norm_w = np.clip(norm_w, 0, 1)
                
                x1, y1 = pos[u]
                x2, y2 = pos[v]
                
                arrow = FancyArrowPatch((x1, y1), (x2, y2),
                                         connectionstyle="arc3,rad=0.1",
                                         arrowstyle='->', mutation_scale=20,
                                         linewidth=1 + 8 * norm_w,
                                         color=plt.cm.viridis(norm_w),
                                         alpha=0.8)
                ax_net.add_patch(arrow)
        
        # Labels
        labels = {n: f"{n.replace('_P1', '').replace('_P2', '')}\n({'P1' if '_P1' in n else 'P2'})" 
                  for n in G.nodes()}
        nx.draw_networkx_labels(G, pos, labels, font_size=12, font_weight='bold', ax=ax_net)
        
        ax_net.text(0, 1.1, 'P1 Modalities', ha='center', fontsize=16, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
        ax_net.text(1, 1.1, 'P2 Modalities', ha='center', fontsize=16, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))
        
        # Colorbar
        sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, 
                                    norm=plt.Normalize(vmin=global_mi_min, vmax=global_mi_max))
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax_net, shrink=0.6)
        cbar.set_label('Mutual Information', fontsize=14, fontweight='bold')
        
        ax_net.set_xlim(-0.3, 1.3)
        ax_net.set_ylim(-0.2, 1.3)
        ax_net.axis('off')
        ax_net.set_title(f'P1-P2 Coupling Network\n{cond_name}', fontsize=18, fontweight='bold')
        
        # === Jitter Plot ===
        envelope_data = []
        within_mod_data = []
        cross_mod_data = []
        
        for _, row in cond_data.iterrows():
            mod1, mod2 = row['modality1'], row['modality2']
            mi = row['max_mutual_info']
            
            if np.isnan(mi):
                continue
            
            if mod1 == 'Audio' and mod2 == 'Audio':
                envelope_data.append(mi)
            elif mod1 == mod2:
                within_mod_data.append(mi)
            else:
                cross_mod_data.append(mi)
        
        categories = []
        values = []
        colors = []
        
        if envelope_data:
            categories.extend(['Envelope\nP1-P2'] * len(envelope_data))
            values.extend(envelope_data)
            colors.extend(['gold'] * len(envelope_data))
        
        if within_mod_data:
            categories.extend(['Within-Modality\nP1-P2'] * len(within_mod_data))
            values.extend(within_mod_data)
            colors.extend(['lightgreen'] * len(within_mod_data))
        
        if cross_mod_data:
            categories.extend(['Cross-Modality\nP1-P2'] * len(cross_mod_data))
            values.extend(cross_mod_data)
            colors.extend(['lightcoral'] * len(cross_mod_data))
        
        if values:
            jitter_df = pd.DataFrame({'category': categories, 'mi': values, 'color': colors})
            cat_names = jitter_df['category'].unique()
            
            for i, cat in enumerate(cat_names):
                cat_data = jitter_df[jitter_df['category'] == cat]
                x_jitter = np.random.normal(i, 0.1, len(cat_data))
                ax_jitter.scatter(x_jitter, cat_data['mi'], c=cat_data['color'].iloc[0],
                                  s=60, alpha=0.7, edgecolors='black', linewidth=0.5)
                
                mean_val = cat_data['mi'].mean()
                ax_jitter.plot([i-0.3, i+0.3], [mean_val, mean_val], 'k-', linewidth=3)
            
            ax_jitter.set_xticks(range(len(cat_names)))
            ax_jitter.set_xticklabels(cat_names, fontsize=12, fontweight='bold')
            ax_jitter.set_ylabel('Mutual Information', fontsize=14, fontweight='bold')
            ax_jitter.set_ylim(global_mi_min, global_mi_max)
        
        ax_jitter.set_title(f'MI Distribution by Type\n{cond_name}', fontsize=16, fontweight='bold')
        ax_jitter.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        safe_cond = cond_name.replace(' ', '_').replace('x', 'vs')
        plt.savefig(os.path.join(output_folder, f'network_jitter_{safe_cond}.png'),
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.savefig(os.path.join(output_folder, f'network_jitter_{safe_cond}.svg'),
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        
        print(f"  ✓ {cond_name}")


def create_all_plots(results_df, output_folder='coupling_plots/'):
    """
    Create all MI-based coupling analysis plots.
    """
    print("\n" + "="*60)
    print("CREATING MUTUAL INFORMATION COUPLING PLOTS")
    print("="*60)
    
    os.makedirs(output_folder, exist_ok=True)
    
    create_modality_overview_plots(results_df, output_folder)
    create_cross_modality_matrices(results_df, output_folder)
    create_audio_vs_modality_scatters(results_df, output_folder)
    create_network_plots(results_df, output_folder)
    
    print("\n" + "="*60)
    print("✓ All plots created successfully!")
    print("="*60)

# running for all combinations

In [ ]:
# Configuration
merged_folder = '../4a_PROCESSED/merged_filteredtimeseries/'
output_file = './p1_p2_coupling_statistics.csv'
output_folder = 'coupling_plots/'
overwrite = True

# Calculate coupling statistics
if overwrite or not os.path.exists(output_file):
    results_df = calculate_p1_p2_coupling_stats_parallel_byfile(
        merged_folder, output_file, n_jobs=-1
    )
else:
    print(f"Loading existing results from {output_file}")
    results_df = pd.read_csv(output_file)

# Create plots
if not results_df.empty:
    create_all_plots(results_df, output_folder)
    
    print("\nExample results:")
    print(results_df.head())
    
    print("\nVariable pair types found:")
    print(results_df['variable_pair_type'].value_counts())

Using 24 CPU cores (file-level parallelization)
Found 20 files to process
Processing 20 files...


[Parallel(n_jobs=24)]: Using backend LokyBackend with 24 concurrent workers.
